In [0]:
import requests
import pandas as pd
import time
import os
import calendar

In [0]:
dbutils.widgets.text("bronze_catalog","dbr_dev")
dbutils.widgets.text("bronze_schema", "artemzharkov10_bronze")

dbutils.widgets.text("silver_catalog","dbr_dev")
dbutils.widgets.text("silver_schema","artemzharkov10_silver")

BRONZE_CATALOG = dbutils.widgets.get("bronze_catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")

SILVER_CATALOG = dbutils.widgets.get("silver_catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

SOURCE_PATH = f"/Volumes/{BRONZE_CATALOG}/{BRONZE_SCHEMA}/raw_data/weather_history"

In [0]:
dbutils.fs.mkdirs(SOURCE_PATH)

In [0]:
grid_df = spark.sql(f"""
    SELECT DISTINCT 
        ROUND(gps_x, 2) AS lat, 
        ROUND(gps_y, 2) AS lon,
        CAST(accident_date AS DATE) AS acc_date
    FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.silver_sewik_accidents
    WHERE gps_x IS NOT NULL AND gps_y IS NOT NULL AND accident_date IS NOT NULL
""")
locations = grid_df.collect()

In [0]:
URL = "https://archive-api.open-meteo.com/v1/archive"
HOURLY_PARAMS = "temperature_2m,precipitation,wind_speed_10m,weather_code"

In [0]:
total_locations = len(locations)
print(f"Всего уникальных событий (координата + дата) для обработки: {total_locations}")

for index, row in enumerate(locations, 1):
    lat = row['lat']
    lon = row['lon']
    
    end_date_obj = row['acc_date'] 
    start_date_obj = end_date_obj - timedelta(days=1)
    
    start_date_str = start_date_obj.strftime('%Y-%m-%d')
    end_date_str = end_date_obj.strftime('%Y-%m-%d')
    
    file_path = f"{SOURCE_PATH}/weather_{lat}_{lon}_{end_date_str}.csv"
    

    if os.path.exists(file_path):
        print(f"[{index}/{total_locations}] Пропуск: Файл {file_path} уже существует.")
        continue
        
    api_params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date_str,
        "end_date": end_date_str,
        "hourly": HOURLY_PARAMS,
        "timezone": "Europe/Warsaw"
    }
    
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = requests.get(URL, params=api_params, timeout=(10, 15))
            
            if response.status_code == 200:
                data = response.json().get("hourly", {})
                
                df_hourly = pd.DataFrame({
                    "time": data.get("time", []),
                    "temperature_2m": data.get("temperature_2m", []),
                    "precipitation": data.get("precipitation", []),
                    "wind_speed_10m": data.get("wind_speed_10m", []),
                    "weather_code": data.get("weather_code", [])
                })
                
                if not df_hourly.empty:
                    df_hourly['time'] = pd.to_datetime(df_hourly['time'])
                    
                    # Отбор строк, где час делится на 4 без остатка
                    df_filtered = df_hourly[df_hourly['time'].dt.hour % 4 == 0].copy()
                    
                    df_filtered['lat'] = lat
                    df_filtered['lon'] = lon
                    df_filtered['target_acc_date'] = end_date_str
                    
                    # Сохранение отфильтрованных данных (ровно 12 строк: 6 за первый день, 6 за второй)
                    df_filtered.to_csv(file_path, index=False)
                    print(f"[{index}/{total_locations}] Успех lat:{lat}, lon:{lon}, date:{end_date_str} | Записано строк: {len(df_filtered)}")
                
                break
                
            elif response.status_code == 429:
                print(f"[{index}/{total_locations}] Ошибка 429 (Лимит). Ожидание 60с. Попытка {attempt + 1}")
                time.sleep(60)
                continue 
                
            else:
                print(f"[{index}/{total_locations}] Ошибка API {response.status_code}. Причина: {response.text}")
                break 
                
        except RequestException as e:
            if attempt < max_retries - 1:
                time.sleep(5) 
            else:
                print(f"[{index}/{total_locations}] Сбой сети. Ошибка: {e}")
    
    time.sleep(1.5)

In [0]:
df_raw = spark.read.csv(f"{SOURCE_PATH}/weather_*.csv", header=True, inferSchema=True)

# 2. Выделение года из даты и запись с группировкой (партиционированием) по годам
TARGET_PATH = f"/Volumes/{BRONZE_CATALOG}/{BRONZE_SCHEMA}/raw_data/weather_by_year"

df_raw.withColumn("year", year(to_date(col("time")))) \
      .write \
      .mode("overwrite") \
      .partitionBy("year") \
      .format("csv") \
      .option("header", "true") \
      .save(TARGET_PATH)

print(f"Данные успешно сгруппированы по годам и сохранены в {TARGET_PATH}")